In [1]:
import os 
os.environ["TAVILY_API_KEY"] = "*******************************"

In [2]:
#!conda install fastmcp -y -vv

In [7]:
%%writefile server.py
import os
from typing import Optional
from fastmcp import FastMCP
from tavily import TavilyClient

# Initialize FastMCP Server
mcp = FastMCP("Tavily Advanced Search Server")

# Define validation constants
VALID_SEARCH_DEPTHS = {"advanced", "basic", "fast", "ultra-fast"}
VALID_TOPICS = {"general", "news", "finance"}
VALID_TIME_RANGES = {"day", "week", "month", "year", "d", "w", "m", "y"}

@mcp.tool()
def web_search(
    query: str, 
    search_depth: str = "basic", 
    max_results: int = 5,
    topic: str = "general", 
    time_range: Optional[str] = None,
    include_answer: bool = False, 
    include_raw_content: bool = False,
    country: Optional[str] = None, 
    include_usage: bool = False
) -> str:
    """Advanced web search tool with parameter validation."""
    
    # 1. API Key Validation
    if not os.environ.get("TAVILY_API_KEY"):
        return "Validation Error: TAVILY_API_KEY environment variable is missing on the server process."

    # 2. Parameter Validation Checks
    errors = []
    if search_depth not in VALID_SEARCH_DEPTHS:
        errors.append(f"Invalid search_depth '{search_depth}' (Choose from: {VALID_SEARCH_DEPTHS})")
    if not (1 <= max_results <= 20):
        errors.append(f"Invalid max_results '{max_results}' (Must be between 1 and 20)")
    if topic not in VALID_TOPICS:
        errors.append(f"Invalid topic '{topic}' (Choose from: {VALID_TOPICS})")
    if time_range and time_range not in VALID_TIME_RANGES:
        errors.append(f"Invalid time_range '{time_range}' (Choose from: {VALID_TIME_RANGES})")

    if errors:
        return "Validation Error: " + ", ".join(errors)

    try:
        # Initialize client cleanly inside the function call to isolate errors
        tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
        
        # Execute search with validated parameters
        response = tavily_client.search(
            query=query, 
            search_depth=search_depth, 
            max_results=max_results,
            topic=topic, 
            time_range=time_range, 
            include_answer=include_answer,
            include_raw_content=include_raw_content, 
            country=country,
            include_usage=include_usage
        )
        
        # 3. Format result safely into a plain string string to prevent FastMCP parsing failures
        results = response.get("results", [])
        if not results:
            return f"Search completed successfully but returned no results for query: '{query}'."
            
        formatted_output = []
        
        if include_answer and response.get("answer"):
            formatted_output.append(f"Direct Answer: {response.get('answer')}\n" + "="*40)
            
        for idx, res in enumerate(results, start=1):
            formatted_output.append(
                f"[{idx}] Title: {res.get('title', 'No Title')}\n"
                f"URL: {res.get('url', 'No URL')}\n"
                f"Snippet: {res.get('content', 'No Content')}\n"
                f"{'-'*40}"
            )
            
        return "\n".join(formatted_output)
        
    except Exception as e:
        # Catch-all to make sure the process doesn't die silently 
        return f"Tavily API Runtime Exception: {str(e)}"

if __name__ == "__main__":
    mcp.run()

Overwriting server.py


In [4]:
#!conda install mcp -y -vv

In [8]:
import sys
import os
import asyncio
import subprocess
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Patch event loop for Jupyter Notebook environments
nest_asyncio.apply()

current_python_exe = sys.executable
print(f"🔧 Targeting target environment: {current_python_exe}")

# Subprocess parameters configuration
server_params = StdioServerParameters(
    command=current_python_exe,
    args=["server.py"],
    env={"TAVILY_API_KEY": os.environ.get("TAVILY_API_KEY")},
    stderr=subprocess.DEVNULL  
)

async def test_validated_workflow():
    print("🤖 Starting background MCP Server and connecting client...")
    try:
        async with stdio_client(server_params, errlog=None) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()
                print("✅ MCP Workflow Engine Client Connected successfully!")
                
                # --- TEST CASE 1: Valid Parameters ---
                print("\n🚀 [Test 1] Dispatching VALID Agent search request...")
                valid_result = await session.call_tool(
                    name="web_search",
                    arguments={
                        "query": "Artificial Intelligence trends 2026",
                        "search_depth": "advanced",
                        "time_range": "month",
                        "max_results": 2
                    }
                )
                print("\n📥 Response content text returned:")
                print(valid_result.content[0].text if hasattr(valid_result.content, '__getitem__') else valid_result.content.text)
                
                # --- TEST CASE 2: Invalid Parameters (Simulating Agent Error) ---
                print("\n❌ [Test 2] Dispatching INVALID parameters...")
                invalid_result = await session.call_tool(
                    name="web_search",
                    arguments={
                        "query": "Stock Market Today",
                        "search_depth": "super-fast",  # Invalid value
                        "max_results": 50,             # Invalid range (> 20)
                    }
                )
                print("\n📥 Error Catch Response returned:")
                print(invalid_result.content[0].text if hasattr(invalid_result.content, '__getitem__') else invalid_result.content.text)
                
    except Exception as e:
        print(f"\n🚨 Client Execution Crash Error: {e}")

# Run client within cell execution thread
await test_validated_workflow()

🔧 Targeting target environment: C:\ProgramData\anaconda3\envs\py__12_pytorch\python.exe
🤖 Starting background MCP Server and connecting client...
✅ MCP Workflow Engine Client Connected successfully!

🚀 [Test 1] Dispatching VALID Agent search request...

📥 Response content text returned:
[1] Title: AI Trends 2026: Future of Artificial Intelligence Explained
URL: https://detectresult.com/artificial-intelligence-trends-2026
Snippet: DetectResult – Global Trending News & Insights

May 3, 2026 · 9:32 AM

artificial intelligence trends 2026 futuristic AI technology concept

# Artificial Intelligence Trends 2026: How AI Is Transforming Industries

By lolita57 · April 23, 2026

# Artificial Intelligence Trends 2026: How AI Is Transforming Industries

Artificial Intelligence (AI) is one of the most powerful technologies shaping the modern world. In 2026, AI continues to expand across multiple industries, improving efficiency and innovation.

This article explores key AI trends and their real-wo

In [9]:
from langchain_ollama import ChatOllama

# Step 2: Connect using the direct local Ollama channel
Model = ChatOllama(
    model="llama3.2:3b-instruct-q4_K_M",
    #model="gemma4:e4b",
    base_url="http://127.0.0.1:11434", # Notice: NO '/v1' path suffix needed here
    temperature=0.0,                   # Recommended baseline sampling for Gemma 4
    num_ctx=65536,                     # Opens the 16k context window for the framework
    num_predict=32768,                  # Provides enough room to write out markdown files
    num_batch=128  ,
    #format="json"
)

In [14]:
from typing import Annotated, Sequence, TypedDict, Literal, Optional
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# 1. State Definition
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

SYSTEM_INSTRUCTION = """
You are an expert Indian Share Market Analysis Agent. Your core task is to evaluate whether a user should BUY, HOLD, or SELL a specific Indian stock right now based on real-time data.

CRITICAL RULES:
1. Always use your `web_search` tool to gather the absolute latest news, corporate actions, and sentiment regarding the target Indian stock.
2. Rely on fresh data. You MUST specify topic='finance' and search_depth='advanced' to get comprehensive analytical reports.
3. If searching for recent updates, specify time_range='week' or time_range='month'.
4. Your final answer must clearly outline: Current Market Sentiments, Critical Technical / Fundamental Triggers, and a definitive action conclusion (BUY, HOLD, or SELL).

Important: When calling the web_search tool, ensure that the country parameter is passed strictly as a 2-letter ISO string (e.g., 'US', 'IN') or a plain text string. Do not structure it as an object or dictionary.
"""

# 2. Dynamic Tool Binding & Graph Execution Function
async def run_langgraph_mcp_agent(user_question: str, langchain_model):
    async with stdio_client(server_params, errlog=None) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as mcp_session:
            await mcp_session.initialize()
            
            # CRITICAL FIX: Use explicit Literals so the local model cannot hallucinate parameter strings
            @tool
            async def web_search(
                query: str, 
                search_depth: Literal["basic", "advanced", "fast", "ultra-fast"] = "advanced", 
                max_results: int = 5,
                topic: Literal["general", "news", "finance"] = "finance", 
                time_range: Optional[Literal["day", "week", "month", "year", "d", "w", "m", "y"]] = "week",
                include_answer: bool = False, 
                include_raw_content: bool = False,
                country: Optional[str] = None, 
                include_usage: bool = False
            ) -> str:
                """
                Searches the web using Tavily API via local MCP Server pipeline. 
                Use this to gather stock prices, news, filings, and finance insights.
                """
                tool_result = await mcp_session.call_tool(
                    name="web_search", 
                    arguments={
                        "query": query, "search_depth": search_depth, "max_results": max_results,
                        "topic": topic, "time_range": time_range, "include_answer": include_answer,
                        "include_raw_content": include_raw_content, "country": country, "include_usage": include_usage
                    }
                )
                
                if isinstance(tool_result.content, list):
                    return "\n".join([item.text for item in tool_result.content if hasattr(item, 'text')])
                elif hasattr(tool_result.content, 'text'):
                    return tool_result.content.text
                return str(tool_result.content)

            tools_map = {"web_search": web_search}
            model_with_tools = langchain_model.bind_tools(list(tools_map.values()))

            # 3. Nodes and Conditional Routing Logic
            def call_model(state: AgentState):
                current_messages = state["messages"]
                if not any(isinstance(m, SystemMessage) for m in current_messages):
                    current_messages = [SystemMessage(content=SYSTEM_INSTRUCTION)] + list(current_messages)
                return {"messages": [model_with_tools.invoke(current_messages)]}

            async def call_tools(state: AgentState):
                last_message = state["messages"][-1]
                tool_messages = []
                for tool_call in last_message.tool_calls:
                    t_name = tool_call["name"]
                    if t_name in tools_map:
                        result_text = await tools_map[t_name].ainvoke(tool_call["args"])
                        tool_messages.append(ToolMessage(content=result_text, tool_call_id=tool_call["id"]))
                return {"messages": tool_messages}

            def route_conditional(state: AgentState):
                return "tools" if state["messages"][-1].tool_calls else END

            # 4. Build and Compile Graph
            workflow = StateGraph(AgentState)
            workflow.add_node("agent", call_model)
            workflow.add_node("tools", call_tools)
            workflow.add_edge(START, "agent")
            workflow.add_conditional_edges("agent", route_conditional, {"tools": "tools", END: END})
            workflow.add_edge("tools", "agent")
            
            app = workflow.compile()

            # 5. Stream Results
            inputs = {"messages": [HumanMessage(content=user_question)]}
            async for chunk in app.astream(inputs, stream_mode="values"):
                last_msg = chunk["messages"][-1]
                if last_msg.content:
                    print(f"\n📬 Update from {type(last_msg).__name__}:")
                    print("-" * 50)
                    print(last_msg.content[:800] + ("..." if len(str(last_msg.content)) > 800 else ""))
                    print("-" * 50)

# Run the compiled LangGraph setup against your pre-configured 'Model'
question = "Should I buy TCS stock on the NSE right now?"
await run_langgraph_mcp_agent(question, Model)


📬 Update from HumanMessage:
--------------------------------------------------
Should I buy TCS stock on the NSE right now?
--------------------------------------------------

📬 Update from ToolMessage:
--------------------------------------------------
Tavily API Runtime Exception: Invalid country. Must be a valid country name from the list of supported countries (https://docs.tavily.com/documentation/api-reference/endpoint/search).
--------------------------------------------------

📬 Update from AIMessage:
--------------------------------------------------
I'll call the web_search tool again with the correct parameters.

{"name": "web_search", "parameters": {"country": "IN", "include_answer": "true", "include_raw_content": "false", "include_usage": "false", "max_results": "10", "query": "TCS stock price and news today", "search_depth": "advanced", "topic": "finance"}}
--------------------------------------------------
